In [ ]:
import ctypes
import time

import pyautogui as pyg


def esc_pressed():
    return bool(ctypes.windll.user32.GetAsyncKeyState(0x1B) & 0x8000)


pyg.PAUSE = 0
DINO_FRONT, SCREEN_RIGHT, Y = 235, 765, 724
FPS, START_SPEED, ACCEL, MAX_SPEED = 60, 19.8, 0.0033, 42.9
DETECT_FRAMES = 12  # 預留約 12 個遊戲影格的反應距離
print("3 秒後開始；偵測點會隨速度前移，按 Esc 停止。")

for _ in range(30):
    if esc_pressed():
        break
    time.sleep(0.1)
else:
    background = pyg.pixel(DINO_FRONT + 10, Y)
    pyg.press("space")
    started = time.perf_counter()
    armed = True
    while not esc_pressed():
        frames = (time.perf_counter() - started) * FPS
        speed = min(START_SPEED + ACCEL * frames, MAX_SPEED)
        detect_x = min(DINO_FRONT + round(speed * DETECT_FRAMES), SCREEN_RIGHT)
        color = pyg.pixel(detect_x, Y)
        obstacle = sum(abs(a - b) for a, b in zip(color, background)) > 90

        if obstacle and armed:
            pyg.press("space")
            armed = False
        elif not obstacle:
            armed = True

        time.sleep(0.01)

print("已停止。")
